In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.seasonal import seasonal_decompose, STL
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.arima_process import ArmaProcess
from statsmodels.graphics.gofplots import qqplot
from statsmodels.tsa.stattools import adfuller
from tqdm import tqdm
from itertools import product
from typing import Union

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import warnings
warnings.filterwarnings('ignore')

%matplotlib inline

plt.rcParams['font.sans-serif'] = ['Microsoft YaHei']
plt.rcParams['axes.unicode_minus'] = False

In [ ]:
df = pd.read_csv('D:\\Apy0\\data\\tpmm\\ts_tp_2000_2024_grid_R22_C76.csv')
df_new = pd.DataFrame({
    'date': pd.date_range('2000-01-01', periods=9132, freq='D'),
    'tp': df['pcpmm']*24
})
df_new.set_index('date', inplace=True)
df_week = df_new.resample('W').mean()
df_week

In [ ]:
fig, ax = plt.subplots(figsize=(9,4))

ax.plot(np.arange(len(df_week)), df_week['tp'])
ax.set_xlabel('year')
ax.set_ylabel('tp(mm)')

tick_positions = np.arange(0, len(df_week), 105)
tick_dates = df_week.index[tick_positions]
plt.xticks(ticks=tick_positions, labels=tick_dates, rotation=60, fontsize=9)
tick_labels = tick_dates.strftime('%Y')
ax.set_xticklabels(tick_labels, rotation=45, ha='right')

fig.autofmt_xdate()
plt.tight_layout()

plt.savefig('fig/CH08_F01_peixeiro.png', dpi=300)

In [ ]:
fig, ax = plt.subplots(figsize=(9,4))

ax.plot(np.arange(len(df_week)), df_week['tp'], markevery=np.arange(0, len(df_week), 105), marker='o')
ax.set_xlabel('year')
ax.set_ylabel('tp(mm)')

tick_positions = np.arange(0, len(df_week), 105)
tick_dates = df_week.index[tick_positions]
plt.xticks(ticks=tick_positions, labels=tick_dates, rotation=60, fontsize=9)
tick_labels = tick_dates.strftime('%Y')
ax.set_xticklabels(tick_labels, rotation=45, ha='right')

fig.autofmt_xdate()
plt.tight_layout()

plt.savefig('fig/CH08_F02_peixeiro.png', dpi=300)

In [ ]:
decomposition = STL(df_week['tp'], period=52).fit()

fig, (ax1, ax2, ax3, ax4) = plt.subplots(nrows=4, ncols=1, sharex=True, figsize=(10,8))

ax1.plot(decomposition.observed)
ax1.set_ylabel('Observed')

ax2.plot(decomposition.trend)
ax2.set_ylabel('Trend')

ax3.plot(decomposition.seasonal)
ax3.set_ylabel('Seasonal')

ax4.plot(decomposition.resid)
ax4.set_ylabel('Residuals')

fig.autofmt_xdate()
plt.tight_layout()

plt.savefig('fig/CH08_F04_peixeiro.png', dpi=300)

In [ ]:
# 平稳性检验
ad_fuller_result = adfuller(df_week['tp'])

print(f'ADF Statistic: {ad_fuller_result[0]}')
print(f'p-value: {ad_fuller_result[1]}')

In [ ]:
def optimize_ARIMA(endog: Union[pd.Series, list], order_list: list, d: int) -> pd.DataFrame:
    
    results = []
    
    for order in tqdm(order_list):
        try: 
            model = SARIMAX(endog, order=(order[0], d, order[1]), simple_differencing=False).fit(disp=False)
        except:
            continue
            
        bic = model.bic
        results.append([order, bic])
        
    result_df = pd.DataFrame(results)
    result_df.columns = ['(p,q)', 'BIC']
    
    #Sort in ascending order, lower BIC is better
    result_df = result_df.sort_values(by='BIC', ascending=True).reset_index(drop=True)
    
    return result_df

In [ ]:
def optimize_SARIMA(endog: Union[pd.Series, list], order_list: list, d: int, D: int, s: int) -> pd.DataFrame:
    
    results = []
    
    for order in tqdm(order_list):
        try: 
            model = SARIMAX(
                endog, 
                order=(order[0], d, order[1]),
                seasonal_order=(order[2], D, order[3], s),
                simple_differencing=False).fit(disp=False)
        except:
            continue
            
        bic = model.bic
        results.append([order, bic])
        
    result_df = pd.DataFrame(results)
    result_df.columns = ['(p,q,P,Q)', 'BIC']
    
    #Sort in ascending order, lower BIC is better
    result_df = result_df.sort_values(by='BIC', ascending=True).reset_index(drop=True)
    
    return result_df

In [ ]:
ps = range(0, 4, 1)
qs = range(0, 4, 1)
Ps = range(0, 4, 1)
Qs = range(0, 4, 1)

d = 0
D = 0
s = 52

SARIMA_order_list = list(product(ps, qs, Ps, Qs))

train = df_week['tp'][:1046]

SARIMA_result_df = optimize_SARIMA(train, SARIMA_order_list, d, D, s)
SARIMA_result_df

In [ ]:
ARIMA_model = SARIMAX(train, order=(2,0,1), simple_differencing=False)
ARIMA_model_fit = ARIMA_model.fit(disp=False)

print(ARIMA_model_fit.summary())

In [ ]:
ARIMA_model_fit.plot_diagnostics(figsize=(10,8))

plt.savefig('fig/CH08_F09_peixeiro.png', dpi=300)

In [ ]:
residuals = ARIMA_model_fit.resid
result = acorr_ljungbox(residuals, lags=np.arange(1, 11, 1), return_df=True)
print(result['lb_pvalue'].values)
# 返回的p值数组，检验残差的自相关性，若所有p值均大于0.05，则残差序列无显著自相关性，模型拟合较好。

In [ ]:
test = df_week.iloc[1044:1306]


In [ ]:
ARIMA_pred = ARIMA_model_fit.get_prediction(781, 1043).predicted_mean
test['ARIMA_pred'] = ARIMA_pred
test

In [ ]:
SARIMA_model = SARIMAX(train, order=(2,0,1), seasonal_order=(0,0,0,52), simple_differencing=False)
SARIMA_model_fit = SARIMA_model.fit(disp=False)

print(SARIMA_model_fit.summary())

In [ ]:
SARIMA_model_fit.plot_diagnostics(figsize=(10,8))

plt.savefig('fig/CH08_F12_peixeiro.png', dpi=300)

In [ ]:
residuals = SARIMA_model_fit.resid
result = acorr_ljungbox(residuals, lags=np.arange(1, 11, 1), return_df=True)
print(result['lb_pvalue'].values)
# 返回的p值数组，检验残差的自相关性，若所有p值均大于0.05，则残差序列无显著自相关性，模型拟合较好。

In [ ]:
SARIMA_pred = SARIMA_model_fit.get_prediction(1044, 1306).predicted_mean

test['SARIMA_pred'] = SARIMA_pred
test

In [ ]:
fig, ax = plt.subplots()

ax.plot(df['Month'], df['Passengers'])
ax.plot(test['Passengers'], 'b-', label='actual')
ax.plot(test['naive_seasonal'], 'r:', label='naive seasonal')
ax.plot(test['ARIMA_pred'], 'k--', label='ARIMA(11,2,3)')
ax.plot(test['SARIMA_pred'], 'g-.', label='SARIMA(2,1,1)(1,1,2,12)')

ax.set_xlabel('Date')
ax.set_ylabel('Number of air passengers')
ax.axvspan(132, 143, color='#808080', alpha=0.2)

ax.legend(loc=2)

plt.xticks(np.arange(0, 145, 12), np.arange(1949, 1962, 1))
ax.set_xlim(120, 143)

fig.autofmt_xdate()
plt.tight_layout()

plt.savefig('fig/CH08_F13_peixeiro.png', dpi=300)

In [ ]:
def mape(y_true, y_pred):
    return np.mean(np.abs((y_true - y_pred) / y_true)) * 100

In [ ]:
mape_naive_seasonal = mape(test['Passengers'], test['naive_seasonal'])
mape_ARIMA = mape(test['Passengers'], test['ARIMA_pred'])
mape_SARIMA = mape(test['Passengers'], test['SARIMA_pred'])

print(mape_naive_seasonal, mape_ARIMA, mape_SARIMA)

In [ ]:
fig, ax = plt.subplots()

x = ['naive seasonal', 'ARIMA(11,2,3)', 'SARIMA(2,1,1)(1,1,2,12)']
y = [mape_naive_seasonal, mape_ARIMA, mape_SARIMA]

ax.bar(x, y, width=0.4)
ax.set_xlabel('Models')
ax.set_ylabel('MAPE (%)')
ax.set_ylim(0, 15)

for index, value in enumerate(y):
    plt.text(x=index, y=value + 1, s=str(round(value,2)), ha='center')

plt.tight_layout()

plt.savefig('fig/CH08_F14_peixeiro.png', dpi=300)